## Configuração e Carregamento de Dados

Nesta secção, definimos as funções para carregar os dados de duas fontes diferentes: um conjunto de dados fornecido pelo docente (arquivos CSV) e um conjunto de dados externo do HC3 (formato JSONL). Em seguida, essas funções são utilizadas para preparar um dataset combinado e limpo.

In [13]:
import pandas as pd
import os # Adicionado para manipulação de caminhos e verificação de arquivos locais

def load_docente_data(inputs_path, outputs_path):
    df_in = pd.read_csv(inputs_path, sep='\t') # Especificar delimitador como tab
    df_out = pd.read_csv(outputs_path, sep='\t') # Corrigido para especificar delimitador como tab
    # Usar pd.merge para combinar os dataframes com base na coluna 'ID'
    df = pd.merge(df_in, df_out, on='ID', how='inner')
    # Ajustar para nomes genéricos
    # ATENÇÃO: As colunas 'Text' e 'Label' foram identificadas como as correctas nos arquivos do docente.
    df = df.rename(columns={'Text': 'text', 'Label': 'label'}) # Corrigido para 'Text' e 'Label'
    return df

def load_external_hc3():
    print("Carregando dados do HC3 localmente de 'data/raw/all.jsonl'")

    try:
        # Carregar o arquivo all.jsonl baixado localmente.
        hc3_data_path = "../data/raw/all.jsonl"
        if os.path.exists(hc3_data_path):
            print(f"Carregando {hc3_data_path}...")
            # O arquivo é um JSONL (JSON Lines), então usamos lines=True
            external_df = pd.read_json(hc3_data_path, lines=True)
            # Processar o df externo para o formato desejado (text, label)
            data = []
            for _, row in external_df.iterrows():
                if 'human_answers' in row:
                    for h_ans in row['human_answers']:
                        data.append({'text': h_ans, 'label': 0})
                if 'chatgpt_answers' in row:
                    for ai_ans in row['chatgpt_answers']:
                        data.append({'text': ai_ans, 'label': 1})
            return pd.DataFrame(data)
        else:
            print(f"Arquivo {hc3_data_path} não encontrado. Por favor, certifique-se de que o dataset HC3 (all.jsonl) foi baixado e colocado em 'data/raw/'.")
            print("Retornando DataFrame vazio para HC3.")
            return pd.DataFrame(columns=['text', 'label'])
    except Exception as e:
        print(f"Erro ao carregar ou processar o dataset HC3 localmente: {e}")
        print("Retornando DataFrame vazio para HC3.")
        return pd.DataFrame(columns=['text', 'label'])

def prepare_combined_dataset(inputs_path, outputs_path, save_output_path):
    df_docente = load_docente_data(inputs_path, outputs_path)
    df_external = load_external_hc3()

    # Concatenar e remover duplicados/nulos
    df_full = pd.concat([df_docente, df_external], ignore_index=True)
    df_full = df_full.dropna().drop_duplicates(subset=['text'])

    # Filtrar aproximadamente pelo tamanho indicado pelo professor (opcional/recomendado)
    df_full['word_count'] = df_full['text'].apply(lambda x: len(x.split()))
    # Mantém textos com tamanho razoável para a tarefa
    df_full = df_full[(df_full['word_count'] >= 30) & (df_full['word_count'] <= 300)]

    df_full[['text', 'label']].to_csv(save_output_path, index=False)
    print(f"Dataset final salvo com {len(df_full)} amostras em {save_output_path}")

if __name__ == "__main__":
    # Certifique-se de que o diretório 'data/processed' existe
    os.makedirs("data/processed", exist_ok=True)
    prepare_combined_dataset("../data/raw/dataset1_inputs.csv", "../data/raw/dataset1_outputs.csv", "../data/processed/full_dataset.csv")

Carregando dados do HC3 localmente de 'data/raw/all.jsonl'
Carregando ../data/raw/all.jsonl...
Dataset final salvo com 30 amostras em ../data/processed/full_dataset.csv


## Carregar e Preparar o Dataset Final

Após a preparação dos dados combinados, eles são carregados em um DataFrame pandas para análises posteriores. Esta célula também verifica a distribuição inicial das classes ('Human' e 'AI') para garantir um balanço adequado.

In [ ]:
# Carregar o dataset final
df_final = pd.read_csv('../data/processed/full_dataset.csv')

# Verificar a distribuição de classes
print('Distribuição de classes no dataset final:')
display(df_final['label'].value_counts())

## Cálculo do Tamanho do Texto

Para a análise, é fundamental ter uma métrica para o tamanho de cada texto. Esta célula adiciona uma nova coluna ao DataFrame, `text_length`, que armazena o número de caracteres de cada texto. Em seguida, calcula e exibe a média do tamanho dos textos, agrupados por sua label (Human ou AI).

In [ ]:
df_final['text_length'] = df_final['text'].apply(len)

print('Média do tamanho dos textos por label:')
display(df_final.groupby('label')['text_length'].mean())

## Visualização da Distribuição do Tamanho dos Textos

Esta seção utiliza histogramas para visualizar a distribuição do `text_length`. O primeiro histograma mostra a distribuição geral para todos os textos, enquanto o segundo compara as distribuições de textos humanos e de IA, permitindo uma análise visual das diferenças.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histograma geral da distribuição de tamanhos de texto
plt.figure(figsize=(10, 6))
sns.histplot(df_final['text_length'], bins=20, kde=True)
plt.title('Distribuição Geral do Tamanho dos Textos')
plt.xlabel('Tamanho do Texto (número de caracteres)')
plt.ylabel('Frequência')
plt.show()

# Histograma da distribuição de tamanhos de texto por label
plt.figure(figsize=(12, 6))
sns.histplot(data=df_final, x='text_length', hue='label', bins=20, kde=True, palette='viridis')
plt.title('Distribuição do Tamanho dos Textos por Label (Human vs AI)')
plt.xlabel('Tamanho do Texto (número de caracteres)')
plt.ylabel('Frequência')
# A legenda já é adicionada automaticamente pelo sns.histplot quando 'hue' é usado
plt.show()

## Boxplot do Tamanho dos Textos por Label

Para complementar a visualização anterior, um boxplot é gerado para comparar a distribuição da `text_length` entre os textos classificados como 'Human' e 'AI'. O boxplot fornece uma visão clara da mediana, quartis e outliers para cada categoria.

In [ ]:
# Criar um boxplot comparando o tamanho dos textos por label
plt.figure(figsize=(8, 6))
sns.boxplot(data=df_final, x='label', y='text_length', hue='label', palette='coolwarm', legend=False)
plt.title('Boxplot do Tamanho dos Textos por Label (Human vs AI)')
plt.xlabel('Label')
plt.ylabel('Tamanho do Texto (número de caracteres)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Gráfico de Dispersão (Scatterplot) do Tamanho do Texto por Label

Esta célula cria um gráfico de dispersão para visualizar a relação individual entre o tamanho de cada texto e sua respectiva label. Embora a variável 'label' seja categórica, a visualização ajuda a identificar possíveis agrupamentos ou tendências.

In [ ]:
# Criar um scatterplot relacionando o tamanho do texto e a label
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_final, x='text_length', y='label', hue='label', palette='viridis', alpha=0.7)
plt.title('Gráfico de Dispersão do Tamanho do Texto por Label')
plt.xlabel('Tamanho do Texto (número de caracteres)')
plt.ylabel('Label')
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.show()

## Cálculo da Correlação entre Tamanho do Texto e Label

Para quantificar a relação entre o tamanho do texto e a label, calcula-se a correlação. Como a coluna 'label' é categórica, ela é primeiro convertida para um formato numérico ('Human' = 0, 'AI' = 1) para que a correlação possa ser computada. A correlação resultante indica a força e a direção da associação.

In [12]:
# Converter a coluna 'label' para um formato numérico
# Por exemplo, 'Human' = 0, 'AI' = 1
df_final['label_numeric'] = df_final['label'].map({'Human': 0, 'AI': 1})

# Calcular a correlação entre 'text_length' e a nova coluna 'label_numeric'
correlation = df_final['text_length'].corr(df_final['label_numeric'])

print(f"Correlação entre o tamanho do texto e a label (Human=0, AI=1): {correlation:.4f}")

# Explicar o que o valor de correlação significa
if abs(correlation) < 0.1:
    print("Isso indica uma correlação muito fraca ou inexistente entre o tamanho do texto e a label.")
elif correlation > 0:
    print("Uma correlação positiva sugere que textos mais longos tendem a estar associados com a label de valor mais alto (neste caso, 'AI').")
else:
    print("Uma correlação negativa sugere que textos mais longos tendem a estar associados com a label de valor mais baixo (neste caso, 'Human').")

# Exibir a matriz de correlação completa se houver mais colunas numéricas no futuro
# display(df_final[['text_length', 'label_numeric']].corr())

Correlação entre o tamanho do texto e a label (Human=0, AI=1): 0.3558
Uma correlação positiva sugere que textos mais longos tendem a estar associados com a label de valor mais alto (neste caso, 'AI').
